# 01 · Baseline SLM senza fine-tuning

Pensato per **SageMaker Studio** su `ml.g6e.2xlarge` (L40S 48 GB, bf16). Ogni modello gira in un processo separato (`python -m slm_baseline.run_baseline`) così la memoria GPU viene liberata tra un modello e l'altro.

Confronti prodotti, tutti sullo **stesso split di test** e con le **stesse metriche** che userà la valutazione post fine-tuning:

| Run | Cosa misura |
|---|---|
| `majority`, `random` | pavimento banale |
| `tfidf_logreg` | baseline classica supervisionata (usa il train) |
| SLM `constrained` | zero-shot, output vincolato alle micro ammesse (metrica principale) |
| SLM `free` | zero-shot libero: misura anche quanto il modello segue il formato |
| SLM `fewshot_constrained` | 1 esempio per micro dal train, opzionale |

Prerequisito: split e `run_config` pronti — da `00` o da `python -m slm_baseline.cli prepare --config configs/experiments/....yaml`.

## 0. Setup SageMaker (una volta per kernel)

In [ ]:
# %pip install -q -r ../requirements-sagemaker.txt
# Poi riavviare il kernel.

In [ ]:
import os
# os.environ["HF_TOKEN"] = "hf_..."   # oppure da Secrets Manager / variabile del Space; serve per Gemma
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "nessuna GPU visibile"

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))  # works even without `pip install -e .`

## 1. Config

`CONFIG_PATH` = YAML condiviso col CLI. `USE_EXPERIMENT_YAML = False` → path legacy.

In [ ]:
USE_EXPERIMENT_YAML = True
USE_CLI_CLASSIC = True  # Step 2: `python -m slm_baseline.cli classic` (dopo prepare in 00)
RERUN_CLASSIC = False
CONFIG_PATH = ROOT / "configs" / "experiments" / "holdout_s42_minimal.yaml"

DATA_PATH = ROOT / "data" / "sample" / "df_sample_150.parquet"
SPLITS_PATH = ROOT / "data" / "splits_v1.parquet"
OUT_DIR = ROOT / "outputs" / "baseline"
TEXT_COLUMN = "ocr_surrogates"
EVAL_SPLIT = "test"
SEED = 42
MODELS = [
    "Qwen/Qwen3-1.7B",
    "HuggingFaceTB/SmolLM3-3B",
    "google/gemma-3-4b-it",
]
MODES = ["constrained", "free"]
LIMIT = 50
GPU_MEM = 0.85

import json
from slm_baseline.experiment import load_experiment_config, resolved_dataset_path

if USE_EXPERIMENT_YAML:
    import os
    os.environ.setdefault("TESI_REPO_ROOT", str(ROOT))
    os.environ.setdefault("TESI_DATA_DIR", str(ROOT))
    os.environ.setdefault("TESI_OUTPUT_DIR", str(ROOT / "outputs"))
    _cfg = load_experiment_config(CONFIG_PATH)
    EXP_DIR = ROOT / "outputs" / "experiments" / _cfg.experiment_id
    OUT_DIR = EXP_DIR / "runs"
    SPLITS_PATH = EXP_DIR / "splits.parquet"
    DATA_PATH = resolved_dataset_path(_cfg)
    TEXT_COLUMN = _cfg.dataset.text_column
    EVAL_SPLIT = _cfg.runs.eval_split
    SEED = _cfg.runs.seed
    LIMIT = _cfg.runs.limit
    GPU_MEM = _cfg.runs.gpu_mem
    MODELS = [s["model"] for s in _cfg.runs.slm]
    MODES = []
    for s in _cfg.runs.slm:
        MODES.extend(s["modes"])
    MODES = list(dict.fromkeys(MODES))
    RUN_CONFIG = json.loads((EXP_DIR / "run_config.json").read_text())
    MAX_SEQ_LEN = RUN_CONFIG["max_seq_len"]

    if USE_CLI_CLASSIC:
        import subprocess

        cmd = [sys.executable, "-m", "slm_baseline.cli", "classic", "--config", str(CONFIG_PATH)]
        if RERUN_CLASSIC:
            cmd.append("--rerun")
        proc = subprocess.run(cmd, cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")})
        if proc.returncode != 0:
            raise RuntimeError("cli classic fallito (eseguire prima prepare da notebook 00)")
        print(f"Step 2 classic OK → {OUT_DIR}")
else:
    EXP_DIR = None
    RUN_CONFIG = json.loads((ROOT / "data" / "run_config.json").read_text())
    MAX_SEQ_LEN = RUN_CONFIG["max_seq_len"]
    assert RUN_CONFIG["text_column"] == TEXT_COLUMN, "text_column diverso da quello usato in 00"

## 2. Baseline classiche (CPU, pochi minuti)

Con `USE_CLI_CLASSIC=True` la cella **Config** ha già lanciato Step 2. La cella sotto è **solo legacy** (ri-esegue predict a mano); imposta `USE_CLI_CLASSIC=False` se vuoi il loop inline.

In [ ]:
import json
import pandas as pd
from slm_baseline.reporting import load_runs

if USE_EXPERIMENT_YAML and USE_CLI_CLASSIC:
    summary, _payload = load_runs(OUT_DIR)
    display(summary[["slug", "macro_f1", "accuracy", "n"]].round(4))
else:
    from slm_baseline.classic import predict_majority, predict_random, predict_tfidf_logreg
    from slm_baseline.data import attach_splits, filter_perimeter, load_segments, prepare_text
    from slm_baseline.inference import CHAT_TEMPLATE_KWARGS
    from slm_baseline.reporting import save_run

    segments, _ = filter_perimeter(load_segments(DATA_PATH))
    segments, dropped = prepare_text(segments, TEXT_COLUMN)
    df = attach_splits(segments, pd.read_parquet(SPLITS_PATH))
    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == EVAL_SPLIT].sort_values("record_key").reset_index(drop=True)
    print(f"train={len(train_df)} test={len(test_df)} esclusi per testo vuoto={len(dropped)}")

    manifest = {}
    if USE_EXPERIMENT_YAML:
        manifest = json.loads((EXP_DIR / "manifest.json").read_text())

    base_cols = ["record_key", "document_id", "macro_category", "sub_category"]
    for name, pred in {
        "majority": predict_majority(train_df, test_df),
        "random": predict_random(test_df, seed=SEED),
        "tfidf_logreg": predict_tfidf_logreg(train_df, test_df, seed=SEED),
    }.items():
        common = {
            "model": name,
            "mode": f"classic_{name}",
            "text_column": TEXT_COLUMN,
            "split": EVAL_SPLIT,
            "max_seq_len": RUN_CONFIG.get("max_seq_len"),
            "chat_template_kwargs": CHAT_TEMPLATE_KWARGS,
        }
        if manifest:
            common.update({
                "experiment_id": manifest.get("experiment_id"),
                "config_hash": manifest.get("config_hash"),
                "split_hash": manifest.get("splits_hash"),
                "prompt_version": RUN_CONFIG.get("prompt_version"),
                "comparable_group": manifest.get("splits_hash"),
            })
        m = save_run(OUT_DIR, name, test_df[base_cols].assign(pred=pred.values), common)
        print(f"{name:14s} acc={m['accuracy']:.3f} macro_f1={m['macro_f1']:.3f}")

## 3. SLM zero-shot con vLLM

In [ ]:
import subprocess, sys

def run(model: str, mode: str) -> None:
    if USE_EXPERIMENT_YAML:
        cmd = [
            sys.executable, "-m", "slm_baseline.run_baseline",
            "--experiment-dir", str(EXP_DIR),
            "--model", model, "--mode", mode,
        ]
    else:
        cmd = [
            sys.executable, "-m", "slm_baseline.run_baseline",
            "--model", model, "--mode", mode,
            "--data", str(DATA_PATH), "--splits", str(SPLITS_PATH), "--split", EVAL_SPLIT,
            "--text-column", TEXT_COLUMN, "--max-seq-len", str(MAX_SEQ_LEN),
            "--gpu-mem", str(GPU_MEM), "--seed", str(SEED), "--out-dir", str(OUT_DIR),
        ] + (["--limit", str(LIMIT)] if LIMIT else [])
    env = {**os.environ, "PYTHONPATH": str(ROOT / "src")}
    proc = subprocess.run(cmd, env=env, cwd=ROOT, capture_output=True, text=True)
    print(proc.stdout.strip().splitlines()[-1] if proc.returncode == 0 else proc.stderr[-4000:])

for model in MODELS:
    for mode in MODES:
        run(model, mode)

## 4. Risultati

In [ ]:
from slm_baseline.reporting import load_runs, per_macro_table

summary, runs = load_runs(OUT_DIR)
summary.to_csv(OUT_DIR / "summary.csv", index=False)
summary.round(4)

In [ ]:
per_macro_table(runs, "macro_f1").round(3)

In [ ]:
import matplotlib.pyplot as plt

s = summary.sort_values("macro_f1")
fig, ax = plt.subplots(figsize=(8, 0.45 * len(s) + 1))
ax.barh(s["run"], s["macro_f1"], xerr=[s["macro_f1"] - s["f1_lo"], s["f1_hi"] - s["macro_f1"]], capsize=3)
ax.set_xlabel("macro-F1 (IC 95% bootstrap)"); ax.set_xlim(0, 1)
plt.tight_layout(); plt.show()

## 5. Analisi errori

In [ ]:
from slm_baseline.labels import MICRO_PER_MACRO
from slm_baseline.metrics import confusion_for_macro
from slm_baseline.reporting import load_predictions

RUN = summary.query("mode != 'classic'")["run"].iloc[0] if (summary["mode"] != "classic").any() else summary["run"].iloc[0]
pred = load_predictions(OUT_DIR, RUN)
print(RUN)
for macro in MICRO_PER_MACRO:
    if (pred["macro_category"] == macro).any():
        print(f"\n— {macro}"); display(confusion_for_macro(pred, macro))

In [ ]:
# Errori a campione, con il testo per capire cosa confonde il modello
errors = pred[pred["pred"] != pred["sub_category"]].merge(test_df[["record_key", "text"]], on="record_key")
cols = ["macro_category", "sub_category", "pred"] + (["raw_output"] if "raw_output" in errors else [])
with pd.option_context("display.max_colwidth", 300):
    display(errors.assign(text=errors["text"].str[:300])[cols + ["text"]].head(15))